In [0]:
%sql
CREATE TABLE IF NOT EXISTS news_project.source.offset_meta_data
(
    table_name string,
    offset_timestamp timestamp,
    last_offset int
)
USING DELTA;

MERGE INTO news_project.source.offset_meta_data AS target
USING (SELECT 'raw_news_data' AS table_name) AS source
ON target.table_name = source.table_name
WHEN NOT MATCHED THEN
  INSERT (table_name, offset_timestamp, last_offset)
  VALUES ('raw_news_data', current_timestamp(), 0);

In [0]:
import requests
from pyspark.sql.functions import *
import time

def get_last_offset(table_name):
    offset_meta_data = spark.sql(f'''SELECT last_offset FROM {table_name}
                                    where offset_timestamp = (SELECT max(offset_timestamp) FROM {table_name})''')
    start_offset = offset_meta_data.collect()[0][0]
    
    return start_offset

def fetch_news_batch(api_key, start_offset, limit):
    all_articles = []
    offset = start_offset
    max_records = start_offset + 200

    while offset < max_records:
        url = f"https://api.mediastack.com/v1/news?access_key={api_key}&offset={offset}&limit={limit}"
        try:
            response = requests.get(url, timeout=10)
            response.raise_for_status()
            data = response.json()
        except requests.exceptions.RequestException as e:
            print(f"Error fetching data: {e}")
            break

        if "data" in data:
            all_articles.extend(data["data"])
            print(f"Added {len(data['data'])} articles. Total: {len(all_articles)}")
        else:
            print("No more articles to fetch.")
            break

        offset += limit
        time.sleep(1)

    return all_articles, offset

def save_to_source(df, catalog, schema, table_name):
    df.write.format("delta") \
            .mode("append") \
            .option("delta.enableChangeDataFeed", "true") \
            .saveAsTable(f"{catalog}.{schema}.{table_name}")
    pass

In [0]:
start_offset = get_last_offset('news_project.source.offset_meta_data')
api_key = dbutils.secrets.get(scope = "news-api-scope", key = "db-host")

all_articles, offset = fetch_news_batch(api_key, start_offset, limit = 100)

if all_articles:

    df = spark.createDataFrame(all_articles) \
            .withColumn("ingest_time", current_timestamp()) \
            .withColumn("source_name", lit("Media_Stack_News_API")) \
            .withColumn("article_id", sha2(concat_ws("||", trim(lower(col("source"))), trim(lower(col("title"))), col("published_at")), 256))

    last_offset = offset
    spark.sql(f'''INSERT INTO news_project.source.offset_meta_data
                (table_name, offset_timestamp, last_offset)
                VALUES
                ('raw_news_data', current_timestamp(), {last_offset})
            ''')

    catalog = "news_project"
    schema = "source"
    table_name = "raw_news_data"

    save_to_source(df, catalog, schema, table_name)
else:
    print("No new articles to fetch.")